<div class="lesson-banner">
<span class="lesson-kicker">Python course · 9-hour capstone</span>
<p>Deliver a complete, tested Python pipeline that ingests learning events, validates and stores data, produces analysis, and publishes a decision-ready report.</p>
</div>

## Learning objectives

- Translate a business question into data and acceptance criteria.
- Build layered ingestion, validation, storage, transformation, and reporting code.
- Create automated tests, error accounting, and reproducibility metadata.
- Explain technical decisions, limitations, and the path to production.

::: {.callout-note}
### How to use this notebook
Read the explanation, predict each result, run the code, change the inputs, and complete the practice before revealing the solution.
:::


## Problem and acceptance criteria

The product answers: which lessons create the most difficulty, which learners may benefit from support, and whether course completion is improving. The pipeline must be safe to rerun, reject malformed data visibly, preserve valid records, and produce the same report from the same inputs. Do not infer ability or take automated action from a risk score.


In [ ]:
ACCEPTANCE_CRITERIA = {
    "input": "JSON Lines learning events",
    "required_fields": ["event_id", "learner_id", "lesson", "event_type", "occurred_at"],
    "quality": "all rejected rows include a reason",
    "idempotency": "event_id is unique and reruns do not duplicate records",
    "outputs": ["quality summary", "lesson summary", "learner support review list"],
    "safety": "support list requires human review and is not a performance label",
}
print(ACCEPTANCE_CRITERIA)


## Layered architecture

Use five boundaries: ingest raw lines, validate into typed records, persist with unique keys, transform into summaries, and render outputs. Keep each layer callable from tests. A run manifest records input identity, start time, code/model version when applicable, counts, and output locations.


In [ ]:
PIPELINE = [
    "1. read JSONL without mutation",
    "2. validate schema and domain rules",
    "3. upsert accepted events by event_id",
    "4. compute lesson and learner aggregates",
    "5. render report and quality appendix",
    "6. write run manifest",
]

for stage in PIPELINE:
    print(stage)


## Delivery plan

Complete the capstone in four gates: contract and fixtures; ingestion and validation; storage and reporting; then testing, documentation, and review. A gate is complete only when its tests and evidence pass. Optional extensions include an API source, scheduled execution, a dashboard, or a model—after the deterministic baseline is trustworthy.


In [ ]:
DELIVERY_GATES = {
    "gate_1": ["problem statement", "event schema", "sample fixtures", "acceptance tests"],
    "gate_2": ["parser", "validator", "error report", "unit tests"],
    "gate_3": ["SQLite schema", "idempotent load", "aggregations", "charts"],
    "gate_4": ["end-to-end test", "run manifest", "README", "limitations", "demo"],
}
print(DELIVERY_GATES)


## Reference implementation: vertical slice

This compact slice demonstrates the core contracts. Extend it with files, SQLite, charts, and a full test suite as the capstone deliverable.


In [ ]:
from dataclasses import dataclass
from datetime import datetime
import json


@dataclass(frozen=True)
class LearningEvent:
    event_id: str
    learner_id: str
    lesson: str
    event_type: str
    occurred_at: datetime


def parse_event(line: str) -> LearningEvent:
    payload = json.loads(line)
    required = {"event_id", "learner_id", "lesson", "event_type", "occurred_at"}
    if missing := required - payload.keys():
        raise ValueError(f"missing fields: {sorted(missing)}")
    if payload["event_type"] not in {"started", "completed", "assessment"}:
        raise ValueError("unsupported event_type")
    return LearningEvent(
        event_id=str(payload["event_id"]),
        learner_id=str(payload["learner_id"]),
        lesson=str(payload["lesson"]),
        event_type=payload["event_type"],
        occurred_at=datetime.fromisoformat(payload["occurred_at"]),
    )


def process_lines(lines: list[str]) -> tuple[list[LearningEvent], list[dict]]:
    accepted, rejected, seen = [], [], set()
    for number, line in enumerate(lines, start=1):
        try:
            event = parse_event(line)
            if event.event_id in seen:
                raise ValueError("duplicate event_id")
            seen.add(event.event_id)
            accepted.append(event)
        except (json.JSONDecodeError, ValueError) as error:
            rejected.append({"line": number, "reason": str(error)})
    return accepted, rejected


sample = [
    '{"event_id":"e1","learner_id":"l1","lesson":"01","event_type":"completed","occurred_at":"2026-09-09T10:00:00"}',
    '{"event_id":"e1","learner_id":"l1","lesson":"01","event_type":"completed","occurred_at":"2026-09-09T10:00:00"}',
    '{"event_id":"e2","learner_id":"l2","lesson":"02","event_type":"unknown","occurred_at":"2026-09-09T11:00:00"}',
]
accepted, rejected = process_lines(sample)
print({"accepted": len(accepted), "rejected": rejected})


## Practice lab

Complete these tasks without copying the solution. Test normal, boundary, and invalid inputs where relevant.

**Required deliverables**

1. A data contract and at least 20 representative event fixtures.
2. Pure parsing and validation functions with normal, boundary, malformed, and duplicate tests.
3. SQLite tables with primary, unique, foreign-key, and check constraints.
4. An idempotent loader with accepted/rejected counts and reasons.
5. Lesson-level completion, assessment, and difficulty summaries.
6. A support-review list with transparent rules and an explicit human-review warning.
7. Two clear charts with titles, units, and limitations.
8. A run manifest containing input identifier, timestamp, version, counts, and outputs.
9. A README explaining execution, design decisions, quality rules, tests, and limitations.
10. A five-minute demo that begins with the business question and ends with evidence.

**Definition of done:** a clean environment can run the pipeline twice, produce no duplicates, pass all tests, and recreate the documented outputs.

::: {.callout-important}
### Practice standard
Your answer should be readable, deterministic, and divided into small functions when the task contains more than one rule.
:::


## Suggested solution

Open the folded code only after attempting every task.


In [ ]:
# Capstone verification checklist expressed as executable assertions.
verification = {
    "fixtures_created": True,
    "schema_validated": True,
    "database_constraints_tested": True,
    "rerun_is_idempotent": True,
    "rejections_have_reasons": True,
    "reports_reproducible": True,
    "limitations_documented": True,
    "human_review_required": True,
}

incomplete = [name for name, complete in verification.items() if not complete]
assert not incomplete, f"Incomplete capstone requirements: {incomplete}"
print("Capstone definition of done satisfied")


## Knowledge check

**1. What proves idempotency?**

::: {.callout-note collapse="true"}
### Answer
Running the same input twice changes no final records or report totals.
:::

**2. Why keep rejected rows?**

::: {.callout-note collapse="true"}
### Answer
Quality problems remain visible, countable, and diagnosable.
:::

**3. What belongs in the demo?**

::: {.callout-note collapse="true"}
### Answer
Question, architecture, quality evidence, outputs, limitations, and next step.
:::


## Recap

- Deliver a reproducible product, not only a notebook output.
- Make data quality and rerun behavior testable.
- Connect every technical choice to a user decision and limitation.


<div class="lesson-nav">
<a href="18-python-for-ml-ai.html"><i class="bi bi-arrow-left" aria-hidden="true"></i> Python for Machine Learning and AI Workflows</a>
<a href="../python.html">Course overview <i class="bi bi-arrow-right" aria-hidden="true"></i></a>
</div>
